## Grover's Algorithm (2-qubit)

Grover's algorithm searches an unstructured list of N = 2ⁿ items for a single marked item, using roughly √N oracle queries instead of the N/2 a classical search needs on average. For n=2 (N=4 items), the optimal number of Grover iterations is just 1, since √4 = 2 and the standard formula for optimal iteration count works out to about π/4 · √N ≈ 1.

**Circuit.** Both qubits start in equal superposition over all four basis states. Each Grover iteration applies two steps: the oracle, which flips the sign of the marked state and leaves all others untouched, and the diffusion operator, which reflects the state about the average amplitude, amplifying the marked state's probability while suppressing the rest. Repeating this process rotates the state closer and closer to the marked item; for N=4, a single iteration is already enough to reach very high measurement probability on the marked state.

**Oracle construction.** The oracle marks a target bit string by using X gates to temporarily flip any qubit that should read 0 in the target, applying a controlled-Z gate (which only introduces a sign flip when both qubits are |1⟩), then undoing the X gates. This makes the CZ fire exactly on the target string, regardless of which string it is.

**Results (1024 shots, marked = '11', 1 iteration):** {'11': 1024} — the marked state was measured with 100% probability, confirming a single Grover iteration is sufficient to fully amplify the marked state for N=4.

In [1]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
from qiskit import transpile

def marked_state_oracle(marked: str) -> QuantumCircuit:
    # Flips the sign of the marked basis state, leaves all others untouched.
    # X gates temporarily flip any qubit that should be |0> in the marked state,
    # so the CZ (which fires when both qubits are |1>) triggers only on the marked string.
    qc = QuantumCircuit(2)
    reversed_marked = marked[::-1]  # Qiskit orders qubits little-endian
    for i, bit in enumerate(reversed_marked):
        if bit == '0':
            qc.x(i)
    qc.cz(0, 1)
    for i, bit in enumerate(reversed_marked):
        if bit == '0':
            qc.x(i)  # undo the temporary flip
    return qc

def diffusion_operator() -> QuantumCircuit:
    # Reflects the state about the average amplitude ("inversion about the mean"),
    # amplifying the marked state's probability after the oracle has flipped its sign.
    qc = QuantumCircuit(2)
    qc.h([0, 1])
    qc.append(marked_state_oracle('00').to_gate(), [0, 1])  # flips sign of |00>
    qc.h([0, 1])
    return qc

def build_grover_circuit(marked: str, iterations: int = 1) -> QuantumCircuit:
    qc = QuantumCircuit(2, 2)
    qc.h([0, 1])  # equal superposition over all 4 basis states

    oracle = marked_state_oracle(marked)
    diffusion = diffusion_operator()

    for _ in range(iterations):
        qc.append(oracle.to_gate(), [0, 1])
        qc.append(diffusion.to_gate(), [0, 1])

    qc.measure([0, 1], [0, 1])
    return qc

grover_circuit = build_grover_circuit('11', iterations=1)
print(grover_circuit.draw())

sim = AerSimulator()
compiled = transpile(grover_circuit, sim)
job = sim.run(compiled, shots=1024)
counts = job.result().get_counts(compiled)
print(counts)
plot_histogram(counts)
plt.show()

     ┌───┐┌─────────────┐┌─────────────┐┌─┐   
q_0: ┤ H ├┤0            ├┤0            ├┤M├───
     ├───┤│  circuit-47 ││  circuit-48 │└╥┘┌─┐
q_1: ┤ H ├┤1            ├┤1            ├─╫─┤M├
     └───┘└─────────────┘└─────────────┘ ║ └╥┘
c: 2/════════════════════════════════════╩══╩═
                                         0  1 
{'11': 1024}
